# Econ 8208 Assignment 2

- Author: Yuxuan Zhao
- Date: 2026-03-23

In [10]:
using LinearAlgebra

# Local LQ Approximation

We are given

- a return function $r(x,u)$, where $x \in \mathbb{R}^n$ and $u \in \mathbb{R}^m$,
- a state transition function $g(x,u)$,
- a steady state $(\bar x, \bar u)$.

We want to construct the matrices for the standard local LQ approximation:
$$
\begin{aligned}
r(x,u) &\approx r(\bar x,\bar u) + \hat x' Q \hat x + 2 \hat x' W \hat u + \hat u' R \hat u, \\
\hat x_{t+1} &= A \hat x_t + B \hat u_t, \\
\hat x &= x-\bar x, \quad \hat u = u-\bar u. \\
\end{aligned}
$$


We need to compute the following matrices:
$$
Q = \frac{1}{2} r_{xx}(\bar x,\bar u), \qquad
W = \frac{1}{2} r_{xu}(\bar x,\bar u), \qquad
R = \frac{1}{2} r_{uu}(\bar x,\bar u),
$$
and
$$
A = g_x(\bar x,\bar u), \qquad B = g_u(\bar x,\bar u).
$$



To compute these matrices, we need to evaluate the second-order partial derivatives of $r$ and the first-order partial derivatives of $g$ at the steady state $(\bar x, \bar u)$.

These derivatives may not be available analytically, so we can compute them by center differences.

### Numerical Jacobian approximation

To approximate the partial derivatives, we can use the central difference formula. For each component $f_i(x)$ and variable $x_j$,
$$
\frac{\partial f_i(x)}{\partial x_j}
\approx
\frac{f_i(x + h e_j) - f_i(x - h e_j)}{2h},
$$
where $h$ is a small step size and $e_j$ is the $j$ th unit vector.

### Pseudocode for numerical Jacobian approximation

**Input:** $f, x, h$

**Output:** $J_f(x)$

**Algorithm:**

1. For each variable $x_j$, construct the unit vector $e_j$.
2. Compute
   $$
   \frac{\partial f_i(x)}{\partial x_j}
   \approx
   \frac{f_i(x + h e_j) - f_i(x - h e_j)}{2h}
   $$
   for all $i = 1, \dots, n$.
3. Form the Jacobian matrix $J_f(x)$.
4. Return $J_f(x)$.

### Numerical Hessian approximation

To approximate the second-order partial derivatives, we can use the central difference formula for second derivatives. For each component $f_i(x)$ and variables $x_j$ and $x_k$,
$$
\frac{\partial^2 f_i(x)}{\partial x_j \partial x_k}
\approx
\frac{f_i(x + h e_j + h e_k) - f_i(x + h e_j - h e_k) - f_i(x - h e_j + h e_k) + f_i(x - h e_j - h e_k)}{4h^2}.
$$

### Pseudocode for numerical Hessian approximation

**Input:** $f, x, h$

**Output:** $H_f(x)$

**Algorithm:**

1. For each pair of variables $(x_j, x_k)$, construct the unit vectors $e_j$ and $e_k$.
2. Compute
   $$
   \frac{\partial^2 f_i(x)}{\partial x_j \partial x_k}
   \approx
   \frac{f_i(x + h e_j + h e_k) - f_i(x + h e_j - h e_k) - f_i(x - h e_j + h e_k) + f_i(x - h e_j - h e_k)}{4h^2}
   $$
   for all $i = 1, \dots, n$.
3. Form the Hessian matrix $H_f(x)$.
4. Return $H_f(x)$.


In [11]:
# -------------------------------------------------------
# Numerical Jacobian using central difference
# Input: 
#   f: R^n -> R^m
#   x: n-dimensional vector
#   h: step size for finite difference (default: 1e-6)
# Output: 
#   J_f(x): m x n matrix
# -------------------------------------------------------
function numerical_jacobian(f, x; h=1e-6)
    x = Float64.(collect(x))
    fx = Float64.(collect(f(x)))

    m = length(fx)
    n = length(x)
    J = zeros(m, n)

    for j in 1:n
        e = zeros(n)
        e[j] = 1.0

        f_plus = Float64.(collect(f(x + h * e)))
        f_minus = Float64.(collect(f(x - h * e)))

        J[:, j] = (f_plus - f_minus) / (2.0 * h)
    end

    return J
end

# -------------------------------------------------------
# Numerical Hessian using second-order centered differences
# Input:
#   f: R^n -> R
#   x: n-dimensional vector
#   h: step size for finite difference (default: 1e-6)
# Output:
#   H_f(x): n x n matrix
# -------------------------------------------------------
function numerical_hessian(f, x; h=1e-6)
    x = Float64.(collect(x))
    n = length(x)
    H = zeros(n, n)

    f0 = f(x)

    for i in 1:n
        ei = zeros(n)
        ei[i] = 1.0

        H[i, i] = (f(x + h * ei) - 2.0 * f0 + f(x - h * ei)) / h^2

        for j in (i + 1):n
            ej = zeros(n)
            ej[j] = 1.0

            value = (
                f(x + h * ei + h * ej)
                - f(x + h * ei - h * ej)
                - f(x - h * ei + h * ej)
                + f(x - h * ei - h * ej)
            ) / (4.0 * h^2)

            H[i, j] = value
            H[j, i] = value
        end
    end

    return H
end

# -------------------------------------------------------
# Numerical cross Hessian using second-order centered differences
# Input:
#   f: R^n x R^m -> R
#   x: n-dimensional vector
#   u: m-dimensional vector
#   h: step size for finite difference (default: 1e-6)
# Output:
#   H_cross(x, u): n x m matrix
# -------------------------------------------------------
function numerical_cross_hessian(f, x, u; h=1e-6)
    x = Float64.(collect(x))
    u = Float64.(collect(u))

    n = length(x)
    m = length(u)
    H = zeros(n, m)

    for i in 1:n
        ei = zeros(n)
        ei[i] = 1.0

        for j in 1:m
            ej = zeros(m)
            ej[j] = 1.0

            H[i, j] = (
                f(x + h * ei, u + h * ej)
                - f(x + h * ei, u - h * ej)
                - f(x - h * ei, u + h * ej)
                + f(x - h * ei, u - h * ej)
            ) / (4.0 * h^2)
        end
    end

    return H
end


numerical_cross_hessian (generic function with 1 method)

## Problem 1

We need to compute the following matrices:
$$
Q = \frac{1}{2} r_{xx}(\bar x,\bar u), \qquad
W = \frac{1}{2} r_{xu}(\bar x,\bar u), \qquad
R = \frac{1}{2} r_{uu}(\bar x,\bar u),
$$
and
$$
A = g_x(\bar x,\bar u), \qquad
B = g_u(\bar x,\bar u).
$$


### Pseudocode

**Input:** $r, g, \bar x, \bar u, h$

**Output:** $Q, W, R, A, B$

**Algorithm:**

1. Compute $r_{xx}(\bar x,\bar u), r_{xu}(\bar x,\bar u), r_{uu}(\bar x,\bar u)$ by numerical Hessian approximation
2. Form
   $$
   Q = \frac{1}{2} r_{xx}(\bar x,\bar u), \qquad
   W = \frac{1}{2} r_{xu}(\bar x,\bar u), \qquad
   R = \frac{1}{2} r_{uu}(\bar x,\bar u).
   $$
3. Compute $A = g_x(\bar x,\bar u)$ and $B = g_u(\bar x,\bar u)$ by numerical Jacobian approximation.
4. Return $Q, W, R, A, B$.


In [12]:
# -------------------------------------------------------
# Construct local LQ approximation by centered differences
# Input:
#   r    : scalar return function r(x, u)
#   g    : state transition function g(x, u)
#   xbar : steady-state state vector
#   ubar : steady-state control vector
#   h    : step size for finite difference (default: 1e-6)
# Output:
#   Q, W, R, A, B for the local LQ problem
# -------------------------------------------------------
function lq_approximation(r, g, xbar, ubar; h=1e-6)
    xbar = Float64.(collect(xbar))
    ubar = Float64.(collect(ubar))

    # Second derivatives of r by direct second-order centered differences
    r_xx = numerical_hessian(x -> r(x, ubar), xbar; h=h)
    r_xu = numerical_cross_hessian(r, xbar, ubar; h=h)
    r_uu = numerical_hessian(u -> r(xbar, u), ubar; h=h)

    # First derivatives of g
    A = numerical_jacobian(x -> g(x, ubar), xbar; h=h)
    B = numerical_jacobian(u -> g(xbar, u), ubar; h=h)

    Q = 0.5 * r_xx
    W = 0.5 * r_xu
    R = 0.5 * r_uu

    return Q, W, R, A, B
end



lq_approximation (generic function with 1 method)

In [14]:
# ---------------------------------------------------------
# Example Test Case
# r(x, u) = x[1]^2 + 3.0 * x[2] + 2.0 * u[1]
# g(x, u) = [0.8 * x[1] + 0.5 * x[2] + 1.0 * u[1];
#            0.2 * x[1] + 0.9 * x[2] - 0.3 * u[1]]
# Steady state: xbar = [2.0, 4.0], ubar = [5.0]
# ---------------------------------------------------------

r(x, u) = x[1]^2 + 3.0 * x[2] + 2.0 * u[1]
g(x, u) = [
    0.8 * x[1] + 0.5 * x[2] + 1.0 * u[1];
    0.2 * x[1] + 0.9 * x[2] - 0.3 * u[1]
]

xbar = [2.0, 4.0]
ubar = [5.0]
h = 1e-6

# Call the function you defined earlier
Q, W, R, A, B = lq_approximation(r, g, xbar, ubar; h=h)

println("Q = ")
println(Q)

println("W = ")
println(W)

println("R = ")
println(R)

println("A = ")
println(A)

println("B = ")
println(B)


Q = 
[1.000088900582341 0.0; 0.0 0.0]
W = 
[0.0008881784197001252; 0.0;;]
R = 
[0.0;;]
A = 
[0.8000000004670937 0.5000000005139782; 0.20000000011677344 0.9000000005254805]
B = 
[1.000000000139778; -0.29999999995311555;;]
